# Anti Hog — Entrenamiento del predictor (Colab)

Predicción binaria P(control) por celda (fecha, zona, slot 3h).

**Pipeline:**
1. Carga datos (subes `events_geocoded.csv`).
2. Construye dataset de celdas (positivos = control observado, negativos = el resto).
3. Split **temporal** 60/20/20 (train/val/test).
4. Búsqueda de hiperparámetros con **Optuna** (30 trials) → mejor config.
5. Entrenamiento final con **early stopping** sobre val.
6. Evaluación honesta en **test** (que el modelo nunca vio).
7. Calibración + feature importance + confusion matrix.
8. Descarga `predictor.pkl` + `metrics.json` + figuras.

In [ ]:
!pip install -q xgboost workalendar optuna

In [ ]:
from google.colab import files
uploaded = files.upload()   # sube events_geocoded.csv
CSV_NAME = next(iter(uploaded))
print('subido:', CSV_NAME)

In [ ]:
import json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, average_precision_score, log_loss, brier_score_loss,
    f1_score, precision_score, recall_score, confusion_matrix,
    roc_curve, precision_recall_curve, ConfusionMatrixDisplay,
)
from sklearn.calibration import calibration_curve
from xgboost import XGBClassifier
import xgboost as xgb
from workalendar.europe.spain import CanaryIslands
import optuna

CAL = CanaryIslands()
SLOT_HOURS = 3
RANDOM_STATE = 42

def is_holiday(d):
    return CAL.is_working_day(d.to_pydatetime().date()) is False and d.weekday() < 5

def hora_to_slot(h):
    if pd.isna(h): return None
    try:
        return int(str(h).split(':')[0]) // SLOT_HOURS
    except Exception:
        return None

print('xgboost:', xgb.__version__, '| optuna:', optuna.__version__)

## 1. Carga + construcción de celdas

Para cada (fecha, zona, slot_3h) en el rango histórico, etiquetamos `y=1` si hubo al menos un control reportado, `y=0` si no.

In [ ]:
# Esquema de zonas para el modelo: 'zona12' (12 zonas, recomendado) o 'zona' (6 zonas, histórico)
ZONA_COL = 'zona12'

df = pd.read_csv(CSV_NAME)
df = df[(df['es_control'] == True) & df[ZONA_COL].notna()]
df['fecha_evento'] = pd.to_datetime(df['fecha'].fillna(df['pub_date']), errors='coerce', utc=True).dt.tz_localize(None)
df = df.dropna(subset=['fecha_evento'])
df['slot'] = df['hora'].apply(hora_to_slot).fillna(-1).astype(int)
df['fecha_dia'] = df['fecha_evento'].dt.normalize()
print(f'esquema de zonas: {ZONA_COL}')
print('eventos de control geocodificados:', len(df))
print('rango fechas:', df['fecha_dia'].min().date(), '→', df['fecha_dia'].max().date())
print('zonas:', sorted(df[ZONA_COL].unique()))

In [ ]:
positives = set(zip(df['fecha_dia'], df[ZONA_COL], df['slot']))

# --- Detectar el período activo (descartar gaps muertos > 90 días) ---
positive_dates = pd.Series(sorted(df['fecha_dia'].unique()))
gaps = positive_dates.diff()
biggest_gap_idx = gaps.idxmax() if not gaps.empty else 0
biggest_gap = gaps.iloc[biggest_gap_idx] if not gaps.empty else pd.Timedelta(0)
print(f'Mayor gap entre eventos: {biggest_gap.days} días '
      f'(de {positive_dates.iloc[biggest_gap_idx-1].date()} a {positive_dates.iloc[biggest_gap_idx].date()})')

if biggest_gap.days > 90:
    pre_segment  = positive_dates.iloc[:biggest_gap_idx]
    post_segment = positive_dates.iloc[biggest_gap_idx:]
    if len(pre_segment) >= len(post_segment):
        active_start, active_end = pre_segment.iloc[0], pre_segment.iloc[-1]
        print(f'  → conservamos segmento PRE-gap: {active_start.date()} → {active_end.date()}  ({len(pre_segment)} días activos)')
    else:
        active_start, active_end = post_segment.iloc[0], post_segment.iloc[-1]
        print(f'  → conservamos segmento POST-gap: {active_start.date()} → {active_end.date()}  ({len(post_segment)} días activos)')
else:
    active_start, active_end = positive_dates.iloc[0], positive_dates.iloc[-1]
    print(f'No hay gap grande, usamos todo: {active_start.date()} → {active_end.date()}')

all_dates = pd.date_range(active_start, active_end, freq='D')
all_zonas = sorted(df[ZONA_COL].dropna().unique())
all_slots = list(range(8))   # slot=-1 (hora desconocida) se descarta como celda negativa

rows = []
for date in all_dates:
    for zona in all_zonas:
        for slot in all_slots:
            y = 1 if (date, zona, slot) in positives else 0
            rows.append((date, zona, slot, y))

cells = pd.DataFrame(rows, columns=['fecha','zona','slot','y'])
cells['dia_semana'] = cells['fecha'].dt.dayofweek
cells['mes'] = cells['fecha'].dt.month
cells['anio'] = cells['fecha'].dt.year
cells['es_finde'] = (cells['dia_semana'] >= 5).astype(int)
cells['es_festivo'] = cells['fecha'].apply(is_holiday).astype(int)
cells['vispera_festivo'] = cells['fecha'].apply(lambda d: is_holiday(d + pd.Timedelta(days=1))).astype(int)
print(f'\ntotal celdas en período activo: {len(cells):,} | tasa positivos: {cells["y"].mean():.4%}')

## 2. Split temporal 60/20/20

Train < cutoff1 ≤ Val < cutoff2 ≤ Test. El test no se usa hasta el final.

In [ ]:
cutoff1 = cells['fecha'].quantile(0.60, interpolation='nearest')
cutoff2 = cells['fecha'].quantile(0.80, interpolation='nearest')
train = cells[cells['fecha'] < cutoff1]
val   = cells[(cells['fecha'] >= cutoff1) & (cells['fecha'] < cutoff2)]
test  = cells[cells['fecha'] >= cutoff2]

feat_cols = ['dia_semana','mes','anio','slot','es_finde','es_festivo','vispera_festivo']
tr_X = pd.get_dummies(train[feat_cols + ['zona']], columns=['zona'])
va_X = pd.get_dummies(val  [feat_cols + ['zona']], columns=['zona']).reindex(columns=tr_X.columns, fill_value=0)
te_X = pd.get_dummies(test [feat_cols + ['zona']], columns=['zona']).reindex(columns=tr_X.columns, fill_value=0)
tr_y, va_y, te_y = train['y'].values, val['y'].values, test['y'].values

print(f'train: {len(train):>7,}  ({train.fecha.min().date()} → {train.fecha.max().date()})  pos={tr_y.mean():.4%}')
print(f'val:   {len(val):>7,}  ({val.fecha.min().date()} → {val.fecha.max().date()})  pos={va_y.mean():.4%}')
print(f'test:  {len(test):>7,}  ({test.fecha.min().date()} → {test.fecha.max().date()})  pos={te_y.mean():.4%}')

## 3. Búsqueda de hiperparámetros con Optuna (30 trials)

Maximiza **PR-AUC** en validación (mejor métrica que ROC-AUC cuando hay desbalance fuerte).

In [ ]:
pos_w = (1 - tr_y.mean()) / max(tr_y.mean(), 1e-6)
print('scale_pos_weight base:', round(pos_w, 1))

def objective(trial):
    params = dict(
        n_estimators       = 2000,                              # cota alta — early stopping lo recorta
        max_depth          = trial.suggest_int('max_depth', 3, 10),
        learning_rate      = trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        subsample          = trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree   = trial.suggest_float('colsample_bytree', 0.6, 1.0),
        min_child_weight   = trial.suggest_int('min_child_weight', 1, 20),
        reg_lambda         = trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        gamma              = trial.suggest_float('gamma', 0.0, 5.0),
        scale_pos_weight   = trial.suggest_float('scale_pos_weight', pos_w*0.3, pos_w*3.0),
        tree_method        = 'hist',
        eval_metric        = 'aucpr',
        early_stopping_rounds = 50,
        random_state       = RANDOM_STATE,
        n_jobs             = -1,
    )
    m = XGBClassifier(**params)
    m.fit(tr_X, tr_y, eval_set=[(va_X, va_y)], verbose=False)
    proba = m.predict_proba(va_X)[:,1]
    return average_precision_score(va_y, proba)

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=30, show_progress_bar=True)

print('\nMejor PR-AUC val:', round(study.best_value, 4))
print('Mejores hiperparámetros:')
for k,v in study.best_params.items():
    print(f'  {k}: {v}')

## 4. Entrenamiento final con early stopping

In [ ]:
best_params = dict(study.best_params)
best_params.update(dict(
    n_estimators=2000, tree_method='hist', eval_metric='aucpr',
    early_stopping_rounds=50, random_state=RANDOM_STATE, n_jobs=-1,
))
model = XGBClassifier(**best_params)
model.fit(tr_X, tr_y, eval_set=[(va_X, va_y)], verbose=False)
print('best_iteration:', model.best_iteration)
print('arboles usados:', model.best_iteration + 1)

## 5. Threshold óptimo (sobre val) y evaluación en test

In [ ]:
va_proba = model.predict_proba(va_X)[:,1]
te_proba = model.predict_proba(te_X)[:,1]

# Buscar threshold que maximiza F1 en VAL
prec, rec, ths = precision_recall_curve(va_y, va_proba)
f1s = 2 * prec * rec / np.maximum(prec + rec, 1e-9)
best_idx = int(np.argmax(f1s[:-1]))
best_threshold = float(ths[best_idx])
print(f'threshold óptimo (val, max F1): {best_threshold:.4f}  (F1_val={f1s[best_idx]:.4f})')

te_pred = (te_proba >= best_threshold).astype(int)

metrics = {
    'roc_auc_test':     float(roc_auc_score(te_y, te_proba)),
    'pr_auc_test':      float(average_precision_score(te_y, te_proba)),
    'log_loss_test':    float(log_loss(te_y, te_proba, labels=[0,1])),
    'brier_test':       float(brier_score_loss(te_y, te_proba)),
    'f1_test':          float(f1_score(te_y, te_pred, zero_division=0)),
    'precision_test':   float(precision_score(te_y, te_pred, zero_division=0)),
    'recall_test':      float(recall_score(te_y, te_pred, zero_division=0)),
    'threshold':        best_threshold,
    'best_iteration':   int(model.best_iteration),
    'baseline_rate_train': float(tr_y.mean()),
    'baseline_rate_test':  float(te_y.mean()),
    'cutoff_train_val': str(cutoff1.date()),
    'cutoff_val_test':  str(cutoff2.date()),
    'n_train': int(len(train)), 'n_val': int(len(val)), 'n_test': int(len(test)),
    'best_params': study.best_params,
    'features':    list(tr_X.columns),
}
for k,v in metrics.items():
    if isinstance(v, float): print(f'  {k:22s} {v:.4f}')
    elif isinstance(v, (list, dict)): continue
    else: print(f'  {k:22s} {v}')

## 6. Plots: ROC, PR, calibración, feature importance, confusion matrix

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# ROC
fpr, tpr, _ = roc_curve(te_y, te_proba)
axes[0,0].plot(fpr, tpr, label=f"AUC={metrics['roc_auc_test']:.3f}")
axes[0,0].plot([0,1],[0,1],'k--', alpha=.3)
axes[0,0].set_xlabel('FPR'); axes[0,0].set_ylabel('TPR'); axes[0,0].set_title('ROC (test)'); axes[0,0].legend()

# PR
prec_t, rec_t, _ = precision_recall_curve(te_y, te_proba)
axes[0,1].plot(rec_t, prec_t, label=f"AP={metrics['pr_auc_test']:.3f}")
axes[0,1].axhline(metrics['baseline_rate_test'], color='r', ls='--', alpha=.4, label=f"baseline={metrics['baseline_rate_test']:.4f}")
axes[0,1].set_xlabel('Recall'); axes[0,1].set_ylabel('Precision'); axes[0,1].set_title('Precision-Recall (test)'); axes[0,1].legend()

# Calibración
frac_pos, mean_pred = calibration_curve(te_y, te_proba, n_bins=10, strategy='quantile')
axes[0,2].plot(mean_pred, frac_pos, 'o-', label='modelo')
axes[0,2].plot([0,1],[0,1],'k--', alpha=.3, label='perfecta')
axes[0,2].set_xlabel('P predicha'); axes[0,2].set_ylabel('frac. positivos real'); axes[0,2].set_title('Calibración (test, quantile)'); axes[0,2].legend()

# Confusion matrix
cm = confusion_matrix(te_y, te_pred)
ConfusionMatrixDisplay(cm, display_labels=['no control','control']).plot(ax=axes[1,0], colorbar=False)
axes[1,0].set_title(f'Confusion matrix (thr={best_threshold:.3f})')

# Feature importance (top 20)
imp = pd.Series(model.feature_importances_, index=tr_X.columns).sort_values(ascending=True).tail(20)
imp.plot.barh(ax=axes[1,1])
axes[1,1].set_title('Feature importance (gain, top 20)')

# Histograma de probas por clase
axes[1,2].hist(te_proba[te_y==0], bins=40, alpha=.5, label='clase 0', density=True)
axes[1,2].hist(te_proba[te_y==1], bins=40, alpha=.5, label='clase 1', density=True)
axes[1,2].axvline(best_threshold, color='red', ls='--', label=f'thr={best_threshold:.2f}')
axes[1,2].set_xlabel('P(control)'); axes[1,2].set_ylabel('densidad'); axes[1,2].set_title('Distribución probas por clase'); axes[1,2].legend()

plt.tight_layout()
plt.savefig('train_report.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Guardar modelo + métricas + descarga

In [ ]:
with open('predictor.pkl','wb') as f:
    pickle.dump({'model': model, 'features': list(tr_X.columns), 'threshold': best_threshold}, f)
with open('metrics.json','w') as f:
    json.dump(metrics, f, indent=2)

from google.colab import files
files.download('predictor.pkl')
files.download('metrics.json')
files.download('train_report.png')